In [ ]:
import os
import random
import shutil
import tarfile
import urllib.request

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

In [ ]:
DATA_URLS = [
    "http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz",
    "https://people.eecs.berkeley.edu/~tinghuiz/projects/pix2pix/datasets/facades.tar.gz",
]
if not os.path.exists("facades"):
    last_error = None
    for url in DATA_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=180) as r, open("facades.tar.gz", "wb") as f:
                shutil.copyfileobj(r, f)
            last_error = None
            break
        except Exception as e:
            last_error = e
    if last_error is not None:
        raise last_error
    with tarfile.open("facades.tar.gz") as tar:
        tar.extractall(".")
print(sorted(os.listdir("facades")))
print(len(os.listdir("facades/train")), "train |", len(os.listdir("facades/val")), "val |", len(os.listdir("facades/test")), "test")

In [ ]:
IMG_SIZE = 256
JITTER_SIZE = 286


class FacadesDataset(Dataset):
    def __init__(self, root="facades", split="train"):
        folder = os.path.join(root, split)
        self.files = sorted(
            os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".jpg")
        )
        self.split = split
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        combined = Image.open(self.files[idx]).convert("RGB")
        w, h = combined.size
        target = combined.crop((0, 0, w // 2, h))
        source = combined.crop((w // 2, 0, w, h))
        if self.split == "train":
            source = source.resize((JITTER_SIZE, JITTER_SIZE), Image.BICUBIC)
            target = target.resize((JITTER_SIZE, JITTER_SIZE), Image.BICUBIC)
            x = random.randint(0, JITTER_SIZE - IMG_SIZE)
            y = random.randint(0, JITTER_SIZE - IMG_SIZE)
            source = source.crop((x, y, x + IMG_SIZE, y + IMG_SIZE))
            target = target.crop((x, y, x + IMG_SIZE, y + IMG_SIZE))
            if random.random() < 0.5:
                source = source.transpose(Image.FLIP_LEFT_RIGHT)
                target = target.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            source = source.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            target = target.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
        source = self.normalize(self.to_tensor(source))
        target = self.normalize(self.to_tensor(target))
        return source, target

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=not use_norm)]
        if use_norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        ]
        if use_dropout:
            layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU())
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DownBlock(3, 64, use_norm=False)
        self.d2 = DownBlock(64, 128)
        self.d3 = DownBlock(128, 256)
        self.d4 = DownBlock(256, 512)
        self.d5 = DownBlock(512, 512)
        self.d6 = DownBlock(512, 512)
        self.d7 = DownBlock(512, 512)
        self.d8 = DownBlock(512, 512, use_norm=False)
        self.u1 = UpBlock(512, 512, use_dropout=True)
        self.u2 = UpBlock(1024, 512, use_dropout=True)
        self.u3 = UpBlock(1024, 512, use_dropout=True)
        self.u4 = UpBlock(1024, 512)
        self.u5 = UpBlock(1024, 256)
        self.u6 = UpBlock(512, 128)
        self.u7 = UpBlock(256, 64)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)
        d7 = self.d7(d6)
        d8 = self.d8(d7)
        u1 = self.u1(d8)
        u2 = self.u2(torch.cat([u1, d7], dim=1))
        u3 = self.u3(torch.cat([u2, d6], dim=1))
        u4 = self.u4(torch.cat([u3, d5], dim=1))
        u5 = self.u5(torch.cat([u4, d4], dim=1))
        u6 = self.u6(torch.cat([u5, d3], dim=1))
        u7 = self.u7(torch.cat([u6, d2], dim=1))
        return self.final(torch.cat([u7, d1], dim=1))

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 1),
        )

    def forward(self, source, target):
        return self.model(torch.cat([source, target], dim=1))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 50
BATCH_SIZE = 1
LR = 2e-4
L1_LAMBDA = 100

torch.manual_seed(42)
random.seed(42)

train_loader = DataLoader(FacadesDataset(split="train"), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(FacadesDataset(split="val"), batch_size=5, shuffle=False)
test_loader = DataLoader(FacadesDataset(split="test"), batch_size=5, shuffle=False)

generator = UNetGenerator().to(DEVICE)
discriminator = PatchDiscriminator().to(DEVICE)
opt_g = torch.optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()

shutil.rmtree("outputs", ignore_errors=True)
os.makedirs("outputs/progress")
os.makedirs("outputs/test_results")

print(DEVICE)
print(f"generator params: {sum(p.numel() for p in generator.parameters()):,}")
print(f"discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

In [ ]:
fixed_source, fixed_target = next(iter(val_loader))
fixed_source, fixed_target = fixed_source.to(DEVICE), fixed_target.to(DEVICE)


def save_comparison(source, target, path):
    generator.eval()
    with torch.no_grad():
        fake = generator(source)
    grid = torch.cat([source, fake, target], dim=0)
    save_image(grid * 0.5 + 0.5, path, nrow=source.size(0))
    generator.train()

In [ ]:
for epoch in range(1, EPOCHS + 1):
    g_running, d_running = 0.0, 0.0
    for source, target in train_loader:
        source, target = source.to(DEVICE), target.to(DEVICE)

        fake = generator(source)
        pred_real = discriminator(source, target)
        pred_fake = discriminator(source, fake.detach())
        loss_d = 0.5 * (
            bce(pred_real, torch.ones_like(pred_real))
            + bce(pred_fake, torch.zeros_like(pred_fake))
        )
        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        pred_fake = discriminator(source, fake)
        loss_g = bce(pred_fake, torch.ones_like(pred_fake)) + L1_LAMBDA * l1(fake, target)
        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        g_running += loss_g.item()
        d_running += loss_d.item()

    print(f"epoch {epoch:03d}/{EPOCHS} | G loss {g_running / len(train_loader):.3f} | D loss {d_running / len(train_loader):.3f}")
    if epoch == 1 or epoch % 10 == 0:
        save_comparison(fixed_source, fixed_target, f"outputs/progress/epoch_{epoch:03d}.png")

torch.save(generator.state_dict(), "outputs/generator.pth")

In [ ]:
generator.eval()
for i, (source, target) in enumerate(test_loader):
    if i >= 4:
        break
    source, target = source.to(DEVICE), target.to(DEVICE)
    save_comparison(source, target, f"outputs/test_results/batch_{i + 1}.png")
print("saved", len(os.listdir("outputs/test_results")), "test grids")

In [ ]:
archive_path = shutil.make_archive("pix2pix_outputs", "zip", "outputs")
print("created", archive_path)
try:
    from google.colab import files
    files.download("pix2pix_outputs.zip")
except ImportError:
    pass